I. Script d'automatisation et pré-modification du fichier csv : Carburant_prix_fr > Dataset> Data.gouv 

In [1]:
pip install scipy sqlalchemy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\mehal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import scipy.stats as st
import os
from sqlalchemy import create_engine



In [3]:
pip install pymysql


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\mehal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:


# === 1. Télécharger automatiquement le fichier depuis data.gouv ===
url = "https://www.data.gouv.fr/api/1/datasets/r/edd67f5b-46d0-4663-9de9-e5db1c880160"
carburant_prix = pd.read_csv(url, sep=';', low_memory=False)

# === 2. Supprimer les colonnes inutiles ===
colonnes_a_supprimer_1 = [
    'services', 'prix', 'rupture', 'horaires détaillés', 'horaires',
    'Début rupture e85 (si temporaire)', 'Type rupture e85'
]
carburant_prix = carburant_prix.drop(columns=colonnes_a_supprimer_1, errors='ignore')

colonnes_a_supprimer_2 = [
    'Prix GPLc mis à jour le', 'Prix GPLc', 'Début rupture GPLc (si temporaire)',
    'Type rupture GPLc', 'Début rupture e10 (si temporaire)', 'Type rupture e10',
    'Début rupture sp98 (si temporaire)', 'Type rupture sp98',
    'Début rupture gazole (si temporaire)'
]
carburant_prix = carburant_prix.drop(columns=colonnes_a_supprimer_2, errors='ignore')

# === 3. Extraire latitude/longitude depuis la colonne 'geom' ===
carburant_prix[['latitude', 'longitude']] = carburant_prix['geom'].str.split(',', expand=True)
carburant_prix = carburant_prix.drop(columns=['geom'])

# === 4. Nettoyer les noms de colonnes ===
carburant_prix.columns = carburant_prix.columns.str.strip().str.lower().str.replace(' ', '_')

# === 5. Renommer les colonnes ===
carburant_prix = carburant_prix.rename(columns={
    'prix_gazole_mis_à_jour_le': 'prix_gazole_maj',
    'prix_sp95_mis_à_jour_le': 'prix_sp95_maj',
    'prix_e85_mis_à_jour_le': 'prix_e85_maj',
    'prix_e10_mis_à_jour_le': 'prix_e10_maj',
    'prix_sp98_mis_à_jour_le': 'prix_sp98_maj',
    'début_rupture_sp95_(si_temporaire)': 'debut_rupture_sp95',
    'automate_24-24_(oui/non)': 'automate_24_24',
    'services_proposés': 'service_propose',
    'début_rupture_sp95': 'debut_rupture_sp95',
    'département': 'departement'
})

# === 6. Conversion des dates ===
colonnes_dates = [
    "prix_gazole_maj", "prix_sp95_maj", "prix_e85_maj",
    "prix_e10_maj", "prix_sp98_maj", "debut_rupture_sp95"
]

for col in colonnes_dates:
    carburant_prix[col] = pd.to_datetime(carburant_prix[col], errors='coerce').dt.tz_localize(None)

# === 7. Sauvegarde finale ===

# → Sauvegarde sur le Bureau
chemin_bureau = os.path.join(os.path.expanduser("~"), "Desktop", "carburant_prix_nettoye.csv")
carburant_prix.to_csv(chemin_bureau, index=False, sep=';', encoding='utf-8-sig')

# → Sauvegarde dans Google Drive (ou autre chemin)
chemin_drive = r"G:\Mon Drive\Carburant prix\carburant_prix_nettoye.csv"
carburant_prix.to_csv(chemin_drive, index=False, sep=';', encoding='utf-8-sig')

print(f"✅ Fichier mis à jour et enregistré ici :\n📁 Bureau : {chemin_bureau}\n📁 Drive : {chemin_drive}")

## Pour connecter et automatiser le fichier csv mise a j par pyhton et mis sur dbeaver 


from sqlalchemy import create_engine

# === 8. Connexion à ta base MariaDB ===
# 💡 Modifie les infos ci-dessous avec ta configuration réelle
utilisateur = "root"  # ou ton user MariaDB
mot_de_passe = "1212.Y8m"
hote = "localhost"
port = "3306"
base = "carburant_prix"  # nom de ta base dans DBeaver

# Création de la connexion SQLAlchemy
engine = create_engine(f"mysql+pymysql://{utilisateur}:{mot_de_passe}@{hote}:{port}/{base}?charset=utf8mb4")


# === 9. Envoi dans la table 'carburant_prix' ===
# Tu peux changer le nom de la table ici si tu veux
carburant_prix.to_sql(name='carburant_prix', con=engine, if_exists='replace', index=False)

print("✅ Données envoyées dans la base MariaDB (table : carburant_prix)")


✅ Fichier mis à jour et enregistré ici :
📁 Bureau : C:\Users\mehal\Desktop\carburant_prix_nettoye.csv
📁 Drive : G:\Mon Drive\Carburant prix\carburant_prix_nettoye.csv
✅ Données envoyées dans la base MariaDB (table : carburant_prix)


In [5]:
print(carburant_prix[['prix_gazole', 'prix_sp95', 'prix_sp98', 'prix_e10', 'prix_e85']].head())
print(carburant_prix[['prix_gazole_maj', 'prix_sp95_maj', 'prix_sp98_maj']].head())


   prix_gazole  prix_sp95  prix_sp98  prix_e10  prix_e85
0        1.749        NaN      1.881     1.771       NaN
1        1.770        NaN        NaN     1.780       NaN
2        1.689        NaN      1.772     1.679       NaN
3        1.639      1.719        NaN       NaN       NaN
4        1.717      1.790      1.845       NaN       NaN
      prix_gazole_maj       prix_sp95_maj       prix_sp98_maj
0 2026-01-30 07:11:20                 NaT 2026-01-30 07:11:20
1 2025-11-13 15:07:50                 NaT                 NaT
2 2026-01-30 00:01:00                 NaT 2026-01-30 00:01:00
3 2026-01-22 12:03:38 2026-01-22 12:03:38                 NaT
4 2026-01-23 11:03:34 2026-01-30 08:35:21 2026-01-23 11:03:35


In [7]:
carburant_prix.head()


,id,latitude,longitude,code_postal,pop,adresse,ville,prix_gazole_maj,prix_gazole,prix_sp95_maj,...,carburants_disponibles,carburants_indisponibles,carburants_en_rupture_temporaire,carburants_en_rupture_definitive,automate_24_24,service_propose,departement,code_departement,région,code_region
0,13400015,43.284,5.602,13400,R,298 Avenue des Paluds,Aubagne,2026-01-30 07:11:20,1.749,NaT,...,"Gazole,E10,SP98","SP95,E85,GPLc",NaN,SP95;E85;GPLc,Oui,"Toilettes publiques,Boutique alimentaire,Carbu...",Bouches-du-Rhône,13,Provence-Alpes-Côte d'Azur,93.0
1,29460003,48.361,-4.26,29460,R,4 Route de Quimper,Daoulas,2025-11-13 15:07:50,1.770,NaT,...,"Gazole,E10","SP95,E85,GPLc,SP98",NaN,SP95;E85;GPLc;SP98,Oui,"Station de gonflage,Location de véhicule,Piste...",Finistère,29,Bretagne,53.0
2,93420007,48.953965389285,2.533686306402,93420,R,BD ROBERT BALLANGER D115,Villepinte,2026-01-30 00:01:00,1.689,NaT,...,"Gazole,GPLc,E10,SP98","SP95,E85",NaN,SP95;E85,Non,"Boutique alimentaire,Boutique non alimentaire,...",Seine-Saint-Denis,93,Île-de-France,11.0
3,60126001,49.359,2.718,60126,R,202 RUE DE PICARDIE,Longueil-Sainte-Marie,2026-01-22 12:03:38,1.639,2026-01-22 12:03:38,...,"Gazole,SP95","E85,GPLc,E10,SP98",NaN,E85;GPLc;E10;SP98,Oui,"Laverie,Relais colis,Boutique alimentaire,Bout...",Oise,60,Hauts-de-France,32.0
4,73140001,45.218,6.472,73140,R,4 PLACE DE LA VANOISE,Saint-Michel-de-Maurienne,2026-01-23 11:03:34,1.717,2026-01-30 08:35:21,...,"Gazole,SP95,SP98","E85,GPLc,E10",NaN,E85;GPLc;E10,Oui,"Vente de gaz domestique (Butane, Propane),Auto...",Savoie,73,Auvergne-Rhône-Alpes,84.0


In [8]:
mask = carburant_prix['adresse'].str.contains("Gier", case=False, na=False) | \
       carburant_prix['adresse'].str.contains("Giers", case=False, na=False)

carburant_prix[mask][['adresse','ville', 'prix_gazole', 'prix_gazole_maj']]



,adresse,ville,prix_gazole,prix_gazole_maj
2762,Rue du Plat du Gier,L'Horme,1.659,2026-01-23 08:20:59
3941,AUTOROUTE A47 - AIRE DE SAINT ROMAIN EN GIER,Saint-Romain-en-Gier,1.879,2026-01-30 06:21:43
5326,Lieu-dit Les Grangiers,Luc-en-Diois,1.684,2026-01-23 14:38:32
7523,24 BOULEVARD AMBROISE BRUGIERE,Clermont-Ferrand,1.744,2026-01-30 00:01:00
7868,25 RUE LOUIS AMARGIER,Saugues,1.669,2026-01-28 16:57:09
8396,AUTOROUTE A47 - AIRE DU PAYS DU GIERS,Saint-Chamond,1.869,2026-01-30 09:00:00
